# Setup & Import


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

# Các thư viện cơ bản
import pandas as pd
import numpy as np
import string
import re
import html
import time
import warnings
warnings.filterwarnings("ignore")

import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, roc_auc_score, accuracy_score, hamming_loss, precision_recall_fscore_support
from sklearn.multiclass import OneVsRestClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit

import nltk
import optuna
import h5sparse
from sentence_transformers import SentenceTransformer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

from modules.data_cleaning import normalize_text, clean_text_dl

print("Khởi tạo môi trường Local thành công!")


# Load Dataset

In [ ]:
# Đọc file dataset từ thư mục gốc
train_df = pd.read_csv('../train.csv')
test_df = pd.read_csv('../test.csv')

print('Train:', train_df.shape)
print('Test:', test_df.shape)

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
X_train = train_df['comment_text'].values
X_test = test_df['comment_text'].values
y_train = train_df[label_cols]

n_classes = len(label_cols)

print("Number of Null values:", train_df.isnull().sum().values.sum())

counts = train_df[label_cols].sum(axis=1).value_counts().sort_index()
print((counts / len(train_df) * 100).round(2))

multi = (train_df[label_cols].sum(axis=1) > 1).mean()
print("Fraction of rows with multiple labels:", multi)

train_df['is_toxic'] = train_df[label_cols].max(axis=1)
toxic_df = train_df[train_df['is_toxic'] == 1]
clean_df = train_df[train_df['is_toxic'] == 0]
print('Toxic:', toxic_df.shape)
print('Clean:', clean_df.shape)

# Traditional Pipeline - Text Preprocessing

In [ ]:
print("Bắt đầu tiền xử lý cho Traditional Pipeline...")
X_normalize = [normalize_text(text) for text in X_train]
X_test_normalize = [normalize_text(text) for text in X_test]

print("Bản ghi mẫu sau khi xử lý:")
print(X_normalize[0])

# Traditional Pipeline - Feature Extraction (TF-IDF)

In [ ]:
print("Fitting TF-IDF on training data...")

tfidf = TfidfVectorizer(min_df=1, max_features=10000)

X_train_tfidf = tfidf.fit_transform(X_normalize)
print(f"X_train shape: {X_train_tfidf.shape}")

print("Transforming test data...")
X_test_tfidf = tfidf.transform(X_test_normalize)
print(f"X_test shape: {X_test_tfidf.shape}")

print("Saving TF-IDF embeddings...")
# Đảm bảo thư mục features tồn tại ở máy local
os.makedirs('../features', exist_ok=True)

with h5sparse.File("../features/tfidf_train_embeddings.h5", "w") as h5f:
    h5f.create_dataset("X_train_tfidf", data=X_train_tfidf)

with h5sparse.File("../features/tfidf_test_embeddings.h5", "w") as h5f:
    h5f.create_dataset("X_test_tfidf", data=X_test_tfidf)

print("Đã lưu embeddings vào thư mục ../features/")


# Traditional Pipeline - End-to-End Training & Optuna

In [ ]:
y_all = train_df[label_cols]

print("Loading TF-IDF embeddings from .h5 file...")
with h5sparse.File("../features/tfidf_train_embeddings.h5", "r") as h5f:
    X_train_tfidf = h5f["X_train_tfidf"].value

with h5sparse.File("../features/tfidf_test_embeddings.h5", "r") as h5f:
    X_test_tfidf = h5f["X_test_tfidf"].value

if X_train_tfidf.shape[0] != len(y_all):
    raise ValueError(
        "Row mismatch between tfidf_train_embeddings.h5 and train.csv labels."
    )

# Multilabel-stratified splitting
y_all_np = y_all.values
msss_outer = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
fit_idx, temp_idx = next(msss_outer.split(np.zeros(len(y_all_np)), y_all_np))

X_fit = X_train_tfidf[fit_idx]
y_fit = y_all.iloc[fit_idx].reset_index(drop=True)

X_temp = X_train_tfidf[temp_idx]
y_temp = y_all.iloc[temp_idx].reset_index(drop=True)
y_temp_np = y_temp.values

msss_inner = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
tune_idx, report_idx = next(msss_inner.split(np.zeros(len(y_temp_np)), y_temp_np))

X_val_tune = X_temp[tune_idx]
y_val_tune = y_temp.iloc[tune_idx].reset_index(drop=True)

X_val_report = X_temp[report_idx]
y_val_report = y_temp.iloc[report_idx].reset_index(drop=True)

# 1. Optuna objective function
def objective(trial):
    c_val = trial.suggest_float("C", 1e-3, 10.0, log=True)
    penalty_val = trial.suggest_categorical("penalty", ["l2"])
    solver_val = trial.suggest_categorical("solver", ["lbfgs"])

    base_model = LogisticRegression(
        C=c_val, penalty=penalty_val, solver=solver_val,
        class_weight="balanced", max_iter=1500, random_state=42, n_jobs=-1,
    )

    y_fit_np = y_fit.values
    mskf = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []

    for tr_idx, va_idx in mskf.split(np.zeros(len(y_fit_np)), y_fit_np):
        X_tr = X_fit[tr_idx]
        X_va = X_fit[va_idx]
        y_tr = y_fit.iloc[tr_idx]
        y_va = y_fit.iloc[va_idx]

        model = OneVsRestClassifier(base_model)
        model.fit(X_tr, y_tr)

        y_va_proba = model.predict_proba(X_va)
        current_fold_label_f1s = []
        threshold_grid = np.arange(0.1, 0.91, 0.005)

        for idx, label in enumerate(label_cols):
            y_true_label = y_va[label].values
            best_f1 = 0.0
            for threshold in threshold_grid:
                y_pred_t = (y_va_proba[:, idx] >= threshold).astype(int)
                score = f1_score(y_true_label, y_pred_t, zero_division=0)
                if score > best_f1:
                    best_f1 = score
            current_fold_label_f1s.append(best_f1)

        fold_scores.append(np.mean(current_fold_label_f1s))
    return float(np.mean(fold_scores))

# 2. Run Optuna study
print("Starting Optuna Hyperparameter Optimization...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
print("\nOptimization Complete! Best Hyperparameters:", study.best_params)

# 3. Train final optimized model
print("\nTraining final model with best parameters...")
best_params = study.best_params
best_base_model = LogisticRegression(
    C=best_params["C"], penalty=best_params["penalty"], solver=best_params["solver"],
    class_weight="balanced", max_iter=1500, random_state=42, n_jobs=-1,
)
final_multi_label_model = OneVsRestClassifier(best_base_model)
final_multi_label_model.fit(X_fit, y_fit)
print("Final model trained successfully!")

# 4. Tune per-label thresholds on tuning split
print("\nTuning per-label thresholds on threshold-tuning split...")
y_tune_proba = final_multi_label_model.predict_proba(X_val_tune)
threshold_grid = np.arange(0.1, 0.95, 0.005)
best_thresholds = {}

for idx, label in enumerate(label_cols):
    y_true_label = y_val_tune[label].values
    best_t = 0.5
    best_f1 = -1.0
    for threshold in threshold_grid:
        y_pred_label = (y_tune_proba[:, idx] >= threshold).astype(int)
        score = f1_score(y_true_label, y_pred_label, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = float(threshold)
    best_thresholds[label] = best_t
print("Best per-label thresholds:", best_thresholds)

# 5. Evaluate on separate report split
print("\nEvaluating on holdout report split...")
y_report_proba = final_multi_label_model.predict_proba(X_val_report)
y_report_pred = np.column_stack([
    (y_report_proba[:, idx] >= best_thresholds[label]).astype(int)
    for idx, label in enumerate(label_cols)
])

print(classification_report(y_val_report, y_report_pred, target_names=label_cols))
print(f"Macro Precision: {precision_score(y_val_report, y_report_pred, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:    {recall_score(y_val_report, y_report_pred, average='macro', zero_division=0):.4f}")
print(f"Macro F1:        {f1_score(y_val_report, y_report_pred, average='macro', zero_division=0):.4f}")

# 6. Predict on test and save outputs
print("\nRetraining final model on 100% of training data for Kaggle submission...")
final_multi_label_model.fit(X_train_tfidf, y_all)

print("\nGenerating predictions for test TF-IDF features...")
y_test_proba = final_multi_label_model.predict_proba(X_test_tfidf)
y_test_pred = np.column_stack([
    (y_test_proba[:, idx] >= best_thresholds[label]).astype(int)
    for idx, label in enumerate(label_cols)
])

os.makedirs('../models', exist_ok=True)
pd.Series(best_thresholds, name="threshold").to_csv("../models/best_thresholds.csv")

test_proba_df = pd.DataFrame(y_test_proba, columns=label_cols)
if "id" in test_df.columns:
    test_proba_df.insert(0, "id", test_df["id"].values)
test_proba_df.to_csv("../models/test_pred_proba.csv", index=False)

test_binary_df = pd.DataFrame(y_test_pred, columns=label_cols)
if "id" in test_df.columns:
    test_binary_df.insert(0, "id", test_df["id"].values)
test_binary_df.to_csv("../models/test_pred_binary.csv", index=False)

print("Saved thresholds, probabilities and binary predictions to ../models/")


# Deep Learning Pipeline - Text Preprocessing

In [ ]:
print("Bắt đầu tiền xử lý cho Deep Learning Pipeline...")
# Gọi hàm clean_text_dl từ modules
X_clean = [clean_text_dl(text) for text in X_train]
X_test_clean = [clean_text_dl(text) for text in X_test]

print("Bản ghi mẫu sau khi xử lý DL:")
print(X_clean[0])


# Deep Learning Pipeline - Feature Extraction

In [ ]:
model_st = SentenceTransformer('all-distilroberta-v1')

X_train_embds = model_st.encode(X_clean, show_progress_bar=True)
X_test_embds = model_st.encode(X_test_clean, show_progress_bar=True)

print("Embeddings of training set: ", X_train_embds.shape)
print("Embeddings of testing set: ", X_test_embds.shape)

# Lưu embeddings vào thư mục features/
np.save('../features/X_train_embeddings.npy', X_train_embds)
np.save('../features/X_test_embeddings.npy', X_test_embds)
print("Exporting embeddings as .npy file successfully.")


# Deep Learning Pipeline - Training PyTorch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading features...")
X_train_dl = np.load('../features/X_train_embeddings.npy')

# Auto-adjust label count
y_train_dl = train_df[label_cols].values[:X_train_dl.shape[0]]
assert X_train_dl.shape[0] == y_train_dl.shape[0], "Error: Feature and label counts do not match!"

# Smoothed Class Weights
num_pos = y_train_dl.sum(axis=0)
num_neg = len(y_train_dl) - num_pos
smoothed_weights = np.sqrt(num_neg / num_pos)
pos_weight_tensor = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)

X_tensor = torch.tensor(X_train_dl, dtype=torch.float32)
y_tensor = torch.tensor(y_train_dl, dtype=torch.float32)
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)

class DeepLogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DeepLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)
    def forward(self, x):
        return self.linear(x)

dl_model = DeepLogisticRegression(X_train_dl.shape[1], 6).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(dl_model.parameters(), lr=0.01)

epochs = 30
print("\nSTARTING PYTORCH TRAINING ON GPU...")
start_time = time.time()

for epoch in range(epochs):
    dl_model.train()
    total_loss = 0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = dl_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss/len(dataloader):.4f}")

# Lưu model vào thư mục models/
os.makedirs('../models', exist_ok=True)
model_path = '../models/pytorch_logreg_smoothed.pth'
torch.save(dl_model.state_dict(), model_path)
print(f"Training completed in {(time.time() - start_time) / 60:.2f} minutes. Saved to: {model_path}")


# Deep Learning Pipeline - Evaluation

In [ ]:
import json
from pathlib import Path

print("Generating predictions on the test set...")
X_test_dl = np.load('../features/X_test_embeddings.npy')

dl_model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test_dl, dtype=torch.float32).to(device)
    logits = dl_model(X_test_tensor)
    probs = torch.sigmoid(logits)
    y_test_proba_dl = probs.cpu().numpy()

# Auto-adjust ID count to match test subset
test_ids = test_df["id"].values[:X_test_dl.shape[0]]

dl_submission = pd.DataFrame(y_test_proba_dl, columns=label_cols)
dl_submission.insert(0, "id", test_ids)
dl_submission.to_csv("../models/test_pred_proba_dl.csv", index=False)

print("Evaluating comprehensive metrics...")
# Đường dẫn ../test_labels.csv vì file nằm ở thư mục gốc
truth_df = pd.read_csv("../test_labels.csv")
merged_dl_df = pd.merge(truth_df, dl_submission, on="id", suffixes=("_true", "_pred"))

# Remove Kaggle's un-scored rows (-1)
clean_dl_df = merged_dl_df[merged_dl_df["toxic_true"] != -1].copy()

y_true = clean_dl_df[[f"{c}_true" for c in label_cols]].values
y_pred_probs = clean_dl_df[[f"{c}_pred" for c in label_cols]].values

# Binarize predictions using default 0.5 threshold
threshold = 0.5
y_pred_bin = (y_pred_probs >= threshold).astype(int)

# 1. ROC-AUC Calculation
roc_auc_scores_dl = []
valid_aucs = []
for i, col in enumerate(label_cols):
    # Handle small test samples where only 1 class might be present
    if len(np.unique(y_true[:, i])) > 1:
        auc = roc_auc_score(y_true[:, i], y_pred_probs[:, i])
        roc_auc_scores_dl.append(auc)
        valid_aucs.append(auc)
    else:
        roc_auc_scores_dl.append(None)

macro_auc = float(np.mean(valid_aucs)) if valid_aucs else None

# 2. Standard Classification Metrics
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred_bin, average=None, zero_division=0)
subset_acc = float(accuracy_score(y_true, y_pred_bin))
micro_f1 = float(f1_score(y_true, y_pred_bin, average="micro", zero_division=0))
macro_f1 = float(f1_score(y_true, y_pred_bin, average="macro", zero_division=0))
h_loss = float(hamming_loss(y_true, y_pred_bin))

# 3. Exporting to match Traditional Pipeline format
# Lưu vào ../reports/ thay vì gốc để gọn gàng
out_dir = Path("../reports/evaluation_output_dl")
out_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "num_total_rows_evaluated": int(len(y_true)),
    "subset_accuracy": subset_acc,
    "micro_f1": micro_f1,
    "macro_f1": macro_f1,
    "hamming_loss": h_loss,
    "macro_roc_auc": macro_auc,
    "label_columns": label_cols,
    "threshold_mode": "global",
    "threshold_used": threshold
}

per_label_df = pd.DataFrame({
    "label": label_cols,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "roc_auc": roc_auc_scores_dl,
    "support": support.astype(int)
})

report_dict = classification_report(y_true, y_pred_bin, target_names=label_cols, zero_division=0, output_dict=True)

with open(out_dir / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
per_label_df.to_csv(out_dir / "per_label_metrics.csv", index=False)
with open(out_dir / "classification_report.json", "w", encoding="utf-8") as f:
    json.dump(report_dict, f, indent=2)

# 4. Print formatted terminal output
print("\n--- DEEP LEARNING EVALUATION COMPLETE ---")
print(f"Rows evaluated: {len(y_true)}")
print(f"Subset accuracy: {subset_acc:.4f}")
print(f"Micro F1: {micro_f1:.4f} | Macro F1: {macro_f1:.4f}")
print(f"Hamming loss: {h_loss:.4f}")
print(f"Mean (Macro) ROC-AUC: {macro_auc:.4f}\n")

print("Per-Label ROC-AUC:")
for label, auc in zip(label_cols, roc_auc_scores_dl):
    auc_str = f"{auc:.4f}" if auc is not None else "N/A"
    print(f"  {label:<15}: {auc_str}")
print("-" * 40)

print("\n" + classification_report(y_true, y_pred_bin, target_names=label_cols, zero_division=0))

print(f"\nFiles successfully exported to '{out_dir}/':")
print(" - metrics_summary.json")
print(" - per_label_metrics.csv")
print(" - classification_report.json")
